# Model development and specification

# Model development and selection

## Candidate models

We considered eight candidate models, starting with a baseline model and progressively adding interaction terms. The candidate models differed in their inclusion of interactions between biological sex, air temperature, and relative humidity. This approach allowed us to assess whether these interactions improved model predictive performance. An overview of the candidate models and their respective interaction terms is provided in <a href="#tbl-candidate-models">Table 1</a>.

<span id="tbl-candidate-models"></span>

**Table 1. Overview of the candidate models.**

| Model | Interaction terms |
|:-------------------|:---------------------------------------------------|
| `mod_00` | None (baseline) |
| `mod_01` | Air temperature × relative humidity |
| `mod_02` | Sex × air temperature |
| `mod_03` | Sex × relative humidity |
| `mod_04` | Sex × air temperature; air temperature × relative humidity |
| `mod_05` | Sex × relative humidity; air temperature × relative humidity |
| `mod_06` | Sex × air temperature; sex × relative humidity |
| `mod_07` | Sex × air temperature; sex × relative humidity; air temperature × relative humidity |

## Model comparison

We initially compared the predictive performance of the eight candidate models using approximate leave-one-out cross-validation (PSIS-LOO), based on the expected log pointwise predictive density (ELPD) (<a href="#tbl-loo-comparison">Table 2</a>). Higher ELPD values indicate better expected predictive performance. The reliability of the PSIS-LOO estimates was assessed using Pareto-$k$ diagnostics.

<span id="tbl-loo-comparison"></span>

**Table 2. Comparison of candidate models based on approximate leave-one-out cross-validation (LOO).**

| Model    | ELPD diff. | SE diff. | ELPD LOO | SE ELPD | Pareto-$k$ diagnostic |
|:---------|-----------:|---------:|---------:|--------:|:----------------------|
| `mod_01` |      0.000 |    0.000 | -2320.52 |   33.03 | 8 observations \> 0.7 |
| `mod_05` |     -0.562 |    2.381 | -2321.08 |   32.95 | 5 observations \> 0.7 |
| `mod_07` |     -0.568 |    2.619 | -2321.09 |   33.08 | 7 observations \> 0.7 |
| `mod_04` |     -0.917 |    1.179 | -2321.43 |   33.16 | 7 observations \> 0.7 |
| `mod_00` |     -1.987 |    3.829 | -2322.50 |   33.92 | 7 observations \> 0.7 |
| `mod_03` |     -2.242 |    4.019 | -2322.76 |   33.74 | 4 observations \> 0.7 |
| `mod_06` |     -3.596 |    4.114 | -2324.11 |   33.86 | 4 observations \> 0.7 |
| `mod_02` |     -4.178 |    3.861 | -2324.70 |   34.01 | 7 observations \> 0.7 |

The PSIS-LOO comparison ranked `mod_01` highest, with an ELPD of -2320.52 (SE = 33.03). However, the Pareto-$k$ diagnostics indicated potential reliability issues, as all eight models had between 4 and 8 observations with Pareto-$k$ values exceeding 0.7. Therefore, the PSIS-LOO estimates warranted additional assessment.

We consequently performed stratified 10-fold cross-validation, using `condition_id` to stratify the folds. Predictive performance was evaluated on held-out observations, providing an additional assessment of the candidate models. The results are presented in <a href="#tbl-kfold-comparison">Table 3</a>.

<span id="tbl-kfold-comparison"></span>

**Table 3. Comparison of candidate models based on stratified 10-fold cross-validation.**

| Model    | ELPD diff. | SE diff. |     ELPD | SE ELPD |
|:---------|-----------:|---------:|---------:|--------:|
| `mod_01` |      0.000 |    0.000 | -2326.69 |   33.67 |
| `mod_05` |     -0.163 |    2.496 | -2326.86 |   33.48 |
| `mod_04` |     -1.735 |    1.241 | -2328.43 |   33.89 |
| `mod_07` |     -2.214 |    2.816 | -2328.91 |   33.77 |
| `mod_00` |     -3.377 |    3.307 | -2330.07 |   34.56 |
| `mod_03` |     -3.700 |    3.951 | -2330.39 |   34.41 |
| `mod_02` |     -5.361 |    3.654 | -2332.06 |   34.83 |
| `mod_06` |     -5.604 |    4.151 | -2332.30 |   34.62 |

The stratified 10-fold cross-validation results also ranked `mod_01` highest, with an ELPD of -2326.69 (SE = 33.67). However, the differences in predictive performance between the leading models were relatively small compared with their standard errors. In particular, `mod_05` had an ELPD difference of only -0.16 (SE = 2.50), indicating very similar estimated predictive performance to `mod_01`.

## Selected model

Based on the model comparisons, we selected `mod_01` as the final model. This model achieved the highest estimated predictive performance under both PSIS-LOO and stratified 10-fold cross-validation. The consistency between the two approaches was further supported by a very strong positive correlation between their model rankings (Spearman’s $\rho$ = 0.952, $p$ \< 0.001, $n$ = 8).

Although `mod_01` ranked first under both approaches, its predictive performance was not clearly distinguishable from that of some alternative models, particularly `mod_05`. The small differences in ELPD relative to their standard errors suggest that several candidate models had broadly similar predictive performance. Nevertheless, the consistent ranking across both comparison methods supported the selection of `mod_01`.

The selected model included an interaction between air temperature and relative humidity, in addition to the main effects and the condition-varying effects. Its full specification and prior distributions are described in the following sections.

# Model specification

## Multinomial logit model

Thermal sensation votes (TSV) were recorded on a seven-point ordinal scale ranging from $-3$ (cold) to $+3$ (hot), with $0$ representing the comfortable category. The data consist of 720 observations across 72 experimental conditions. Each observation represents the TSV counts of five participants of the same biological sex under a given combination of experimental conditions.

Because individual participant identifiers were unavailable, the model was fitted to aggregated category counts rather than individual-level repeated measurements. The response for observation $i$ is represented by a seven-dimensional count vector:

$$
\mathbf Y_i =
(Y_{i,-3},Y_{i,-2},Y_{i,-1},Y_{i,0},Y_{i,+1},Y_{i,+2},Y_{i,+3})
\tag{1}
$$

where the counts sum to five:

$$
\sum_{k=-3}^{+3}Y_{i,k}=5
\tag{2}
$$

The response vector is modelled using a multinomial likelihood:

<span id="eq-likelihood"></span>

$$
\mathbf Y_i
\sim
\operatorname{Multinomial}
\left(5,\mathbf p_i\right)
\tag{3}
$$

where $\mathbf p_i$ is the vector of probabilities for the seven TSV categories.

The comfortable category ($\mathrm{TSV}=0$) is used as the reference category. Its linear predictor is fixed to zero:

$$
\eta_{i,0}=0
\tag{4}
$$

For the six non-reference categories, the category probabilities are obtained through the softmax transformation:

<span id="eq-softmax"></span>

$$
p_{i,k}
=
\frac{\exp(\eta_{i,k})}
{1+\displaystyle\sum_{\ell\in\mathcal K}\exp(\eta_{i,\ell})},
\qquad
k\in\mathcal K
\tag{5}
$$

where $\mathcal K=\{-3,-2,-1,+1,+2,+3\}$.

The probability of the comfortable category is consequently

<span id="eq-reference-probability"></span>

$$
p_{i,0}
=
\frac{1}
{1+\displaystyle\sum_{\ell\in\mathcal K}
\exp(\eta_{i,\ell})}
\tag{6}
$$

The linear predictor for each non-reference category $k$ is specified as

<span id="eq-linear-predictor"></span>

$$
\begin{aligned}
\eta_{i,k} =& \alpha_{\mathrm{condition}[i],\mathrm{sex}[i],k} + \beta_{\mathrm{condition}[i],\mathrm{t.onset},k}
\operatorname{mo}_k(\mathrm{t.onset}_i) + \beta_{\mathrm{t.day},k}\mathrm{t.day}_i \\
&+ \beta_{\mathrm{temp},k}\mathrm{temp}_i + \beta_{\mathrm{hum},k}\mathrm{hum}_i + \beta_{\mathrm{temp.hum},k}(\mathrm{temp}_i\mathrm{hum}_i)
\end{aligned}
\tag{7}
$$

Here, $k$ indexes the non-reference TSV categories, $\mathrm{sex}[i]$ identifies the biological sex of observation $i$, and $\mathrm{condition}[i]$ identifies its experimental condition. The variable $\mathrm{t.day}_{i}$ represents the time-of-day indicator (afternoon/evening), while $\mathrm{t.onset}_i$ denotes the ordered level of time since onset for observation $i$, and $\operatorname{mo}_k(\mathrm{t.onset}_i)$ represents its category-specific monotonic effect. The variable $\mathrm{temp}_{i}$ is centred air temperature, and $\mathrm{hum}_{i}$ is centred relative humidity expressed in 5% units. The temperature–humidity interaction is represented by their product.

## Multilevel structure

The sex coefficients and the monotonic time-since-onset coefficients are allowed to vary across experimental conditions. The condition-specific effects are decomposed into a constant (population-level) and condition-specific deviations:

<span id="eq-sex-decomposition"></span>

$$
\alpha_{\mathrm{condition}[i],\mathrm{sex}[i],k} = \bar{\alpha}_{\mathrm{sex}[i],k} + v_{\mathrm{condition}[i],\mathrm{sex}[i],k}
\tag{8}
$$

<span id="eq-onset-decomposition"></span>

$$
\beta_{\mathrm{condition}[i],\mathrm{t.onset},k} = \bar{\beta}_{\mathrm{t.onset},k} + v_{\mathrm{condition}[i],\mathrm{t.onset},k}
\tag{9}
$$

Here, $\bar{\alpha}_{\mathrm{sex}[i],k}$ and $\bar{\beta}_{\mathrm{t.onset},k}$ represent the constant (population-level) coefficients for the corresponding sex and monotonic time-since-onset effect, respectively. The terms $v_{\mathrm{condition}[i],\mathrm{sex}[i],k}$ and $v_{\mathrm{condition}[i],\mathrm{t.onset},k}$ represent condition-specific deviations from these constant (population-level) coefficients. This hierarchical structure allows partial pooling across experimental conditions.

The condition-varying effects are jointly modelled. For each condition $j$ (with $j=1,\ldots,72$ indexing the 72 experimental conditions), the vector of deviations is defined as

<span id="eq-random-effect-vector"></span>

$$
\begin{aligned}
\mathbf v_j = (&
v_{j,\mathrm{male},-3},
v_{j,\mathrm{female},-3},
v_{j,\mathrm{t.onset},-3},
\\
&
v_{j,\mathrm{male},-2},
v_{j,\mathrm{female},-2},
v_{j,\mathrm{t.onset},-2},
\\
&
v_{j,\mathrm{male},-1},
v_{j,\mathrm{female},-1},
v_{j,\mathrm{t.onset},-1},
\\
&
v_{j,\mathrm{male},+1},
v_{j,\mathrm{female},+1},
v_{j,\mathrm{t.onset},+1},
\\
&
v_{j,\mathrm{male},+2},
v_{j,\mathrm{female},+2},
v_{j,\mathrm{t.onset},+2},
\\
&
v_{j,\mathrm{male},+3},
v_{j,\mathrm{female},+3},
v_{j,\mathrm{t.onset},+3}
)^\mathsf T
\end{aligned}
\tag{10}
$$

Thus, each condition has a vector of 18 condition-varying effects, jointly modelled according to

<span id="eq-random-effects"></span>

$$
\mathbf v_j
\sim
\operatorname{MVN}
\left(
\mathbf 0_{18},\boldsymbol\Sigma_{18\times18}
\right),
\qquad j=1,\ldots,72
\tag{11}
$$

where $\boldsymbol\Sigma_{18\times18}$ is an $18\times18$ covariance matrix. This structure allows correlations between condition-varying effects across sexes, monotonic onset coefficients, and TSV categories.

### Non-centred parameterisation

For computational efficiency, the multivariate normal distribution of the condition-varying effects is implemented using a non-centred parameterisation. First, the standardised condition effects are defined as

<span id="eq-standard-normal-effects"></span>

$$
\mathbf z_j
\sim
\operatorname{MVN}
\left(
\mathbf 0,\mathbf I_{18}
\right),
\qquad j=1,\ldots,72
\tag{12}
$$

where $\mathbf I_{18}$ is the $18\times18$ identity matrix.

The condition-varying effects are then obtained through the transformation

<span id="eq-non-centred"></span>

$$
\mathbf v_j
=
\mathbf D_{\sigma}
\mathbf L_{\Omega}
\mathbf z_j,
\qquad j=1,\ldots,72
\tag{13}
$$

where $\mathbf D_{\sigma}$ is a diagonal matrix containing the group-level standard deviations, and $\mathbf L_{\Omega}$ is the Cholesky factor of the correlation matrix $\boldsymbol\Omega$.

The covariance matrix is therefore expressed as

<span id="eq-covariance-decomposition"></span>

$$
\boldsymbol\Sigma
=
\mathbf D_{\sigma}
\boldsymbol\Omega
\mathbf D_{\sigma},
\qquad
\boldsymbol\Omega
=
\mathbf L_{\Omega}
\mathbf L_{\Omega}^{\mathsf T}
\tag{14}
$$

This parameterisation ensures that the condition-varying effects follow the multivariate normal distribution specified in <a href="#eq-random-effects">Equation (11)</a>.

## Monotonic effect of time since onset

The effect of time since onset is modelled as a monotonic effect rather than assuming equal spacing between its ordered levels. Let $D$ denote the number of ordered levels of time since onset, and let $\mathrm{t.onset}_i\in\{1,\ldots,D\}$ denote the corresponding level for observation $i$. For each non-reference TSV category $k$, the monotonic effect is defined as

<span id="eq-monotonic-effect"></span>

$$
\operatorname{mo}_k(\mathrm{t.onset}_i)
=
(D-1)
\sum_{d=1}^{D-1}
\zeta_{k,d}\mathbb I(\mathrm{t.onset}_i>d)
\tag{15}
$$

where $\mathbb I(\cdot)$ is the indicator function and $\boldsymbol\zeta_k$ is a category-specific simplex vector.

The simplex weights satisfy

<span id="eq-simplex-constraints"></span>

$$
\zeta_{k,d}\geq0,
\qquad
\sum_{d=1}^{D-1}\zeta_{k,d}=1
\tag{16}
$$

The simplex weights determine how the overall monotonic effect is distributed across successive levels of time since onset. The scaling factor $D-1$ ensures that the monotonic effect ranges from zero at the first level to $D-1$ at the final level.

Each non-reference TSV category has its own simplex vector:

<span id="eq-dirichlet-prior"></span>

$$
\boldsymbol\zeta_k
\sim
\operatorname{Dirichlet}
(\boldsymbol\gamma_k),
\qquad
k\in\mathcal K
\tag{17}
$$

where $\boldsymbol\gamma_k$ denotes the vector of prior concentration parameters for category $k$.

The simplex weights are category-specific but are shared across experimental conditions. Condition-specific variation in the time-since-onset effect is captured by the corresponding condition-varying regression coefficient in <a href="#eq-onset-decomposition">Equation (9)</a>.

## Prior specification

### Constant (population-level) regression coefficients

#### Biological sex

The sex-specific baseline coefficients have category-dependent prior means:

$$
\begin{aligned}
\bar\alpha_{\mathrm{male}, -3},\bar\alpha_{\mathrm{female}, -3}
&\sim
\operatorname{Normal}(-2.69,2)\\
\bar\alpha_{\mathrm{male}, -2},\bar\alpha_{\mathrm{female}, -2}
&\sim
\operatorname{Normal}(-1.84,2)\\
\bar\alpha_{\mathrm{male}, -1},\bar\alpha_{\mathrm{female}, -1}
&\sim
\operatorname{Normal}(-0.89,2) \\
\bar\alpha_{\mathrm{male}, 1},\bar\alpha_{\mathrm{female}, 1}
&\sim
\operatorname{Normal}(-0.89,2) \\
\bar\alpha_{\mathrm{male}, 2},\bar\alpha_{\mathrm{female}, 2}
&\sim
\operatorname{Normal}(-1.84,2)\\
\bar\alpha_{\mathrm{male}, 3},\bar\alpha_{\mathrm{female}, 3}
&\sim
\operatorname{Normal}(-2.69,2)
\end{aligned}
\tag{18}
$$

#### Time since onset

The monotonic time since onset coefficients have the prior:

$$
\begin{aligned}
\bar\beta_{\mathrm{t.onset}, -3}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, -2}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, -1}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, 1}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, 2}
&\sim
\operatorname{Normal}(0,0.25)\\
\bar\beta_{\mathrm{t.onset}, 3}
&\sim
\operatorname{Normal}(0,0.25)
\end{aligned}
\tag{19}
$$

Each non-reference TSV category has its own simplex parameter:

$$
\begin{aligned}
\boldsymbol{\zeta}_{-3}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{-2}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{-1}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{1}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{2}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\boldsymbol{\zeta}_{3}
\sim
\operatorname{Dirichlet}(2,2,2,2)\\
\end{aligned}
\tag{20}
$$

The Dirichlet prior is symmetric, assigning equal prior concentration to each increment of the monotonic effect.

#### Time of day

The time-of-day coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{t.day}, -3}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, -2}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, -1}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, 1}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, 2}
&\sim
\operatorname{Normal}(0,0.25)\\
\beta_{\mathrm{t.day}, 3}
&\sim
\operatorname{Normal}(0,0.25)
\end{aligned}
\tag{21}
$$

#### Air temperature

The air temperature coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{temp}, -3}
&\sim
\operatorname{Normal}(-1,0.5)\\
\beta_{\mathrm{temp}, -2}
&\sim
\operatorname{Normal}(-1,0.5)\\
\beta_{\mathrm{temp}, -1}
&\sim
\operatorname{Normal}(-1,0.5)\\
\beta_{\mathrm{temp}, 1}
&\sim
\operatorname{Normal}(1,0.5)\\
\beta_{\mathrm{temp}, 2}
&\sim
\operatorname{Normal}(1,0.5)\\
\beta_{\mathrm{temp}, 3}
&\sim
\operatorname{Normal}(1,0.5)
\end{aligned}
\tag{22}
$$

#### Relative humidity

The relative humidity coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{hum}, -3}
&\sim
\operatorname{Normal}(-0.5,0.25)\\
\beta_{\mathrm{hum}, -2}
&\sim
\operatorname{Normal}(-0.5,0.25)\\
\beta_{\mathrm{hum}, -1}
&\sim
\operatorname{Normal}(-0.5,0.25)\\
\beta_{\mathrm{hum}, 1}
&\sim
\operatorname{Normal}(0.5,0.25)\\
\beta_{\mathrm{hum}, 2}
&\sim
\operatorname{Normal}(0.5,0.25)\\
\beta_{\mathrm{hum}, 3}
&\sim
\operatorname{Normal}(0.5,0.25)
\end{aligned}
\tag{23}
$$

#### Air temperature and relative humidity interaction

The air temperature and relative humidity interaction coefficients have the priors:

$$
\begin{aligned}
\beta_{\mathrm{temp.hum}, -3}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, -2}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, -1}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, 1}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, 2}
&\sim
\operatorname{Normal}(0,0.2)\\
\beta_{\mathrm{temp.hum}, 3}
&\sim
\operatorname{Normal}(0,0.2)
\end{aligned}
\tag{24}
$$

### Condition-varying effects

The standard deviations of the condition-varying effects are assigned exponential priors:

$$
\sigma_r
\sim
\operatorname{Exponential}(1),
\qquad r=1,\ldots,18
\tag{25}
$$

where the index $r$ identifies the 18 dimensions of the condition-varying effects vector.

The correlation matrix is assigned an LKJ prior:

$$
\boldsymbol\Omega
\sim
\operatorname{LKJ}(2)
\tag{26}
$$

The standardised effects used in the non-centred parameterisation have independent standard normal priors:

$$
z_{j,r}
\sim
\operatorname{Normal}(0,1),
\qquad j=1,\ldots,72,
\qquad r=1,\ldots,18
\tag{27}
$$